# 03 - Output Structuring (输出结构化)

## 学习目标
- 使用 OpenAI Structured Outputs 和 JSON Schema 强制约束输出格式
- 构建多级回退解析器：json.loads -> regex -> repair -> retry
- 掌握多种解析策略：CommaSeparatedList, Enum, Pydantic
- 了解 Instructor 库的替代方案和优雅降级策略

In [ ]:
# 初始化环境
import os
import re
import json
import inspect
from enum import Enum
from typing import Any, Type, get_origin, get_args
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, ValidationError, field_validator

print("环境已初始化")

---
## 1. 定义 Pydantic 模型（期望的输出结构）

Pydantic 模型定义了输出的"契约"。每个字段都有类型、约束和验证逻辑。

In [ ]:
# ============================================================
# 定义结构化的 Pydantic 输出模型
# ============================================================

class Sentiment(str, Enum):
    """情感枚举：限定输出类别"""
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

class Priority(str, Enum):
    """优先级枚举"""
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class CustomerFeedback(BaseModel):
    """
    客户反馈的结构化模型。
    
    What: 定义我们期望从 LLM 获取的客户反馈分析结果。
    Why: 使用 Pydantic 模型的类型系统和验证约束输出格式。
    When: 任何需要解析 LLM 输出为结构化数据的场景。
    """
    sentiment: Sentiment = Field(description="情感倾向")
    confidence: float = Field(
        ge=0.0, le=1.0,  # 约束范围
        description="置信度，0.0 到 1.0 之间"
    )
    summary: str = Field(
        min_length=5, max_length=200,
        description="反馈摘要，5-200字符"
    )
    keywords: list[str] = Field(
        min_length=1, max_length=10,
        description="关键词列表，1-10个"
    )
    urgency: Priority = Field(
        default=Priority.LOW,
        description="紧急程度"
    )
    rating: int | None = Field(
        default=None, ge=1, le=5,
        description="评分，1-5，如果无法判断则为 None"
    )
    
    @field_validator('keywords')
    @classmethod
    def keywords_not_empty(cls, v: list[str]) -> list[str]:
        """验证关键词不能为空字符串"""
        if any(not kw.strip() for kw in v):
            raise ValueError("关键词不能为空")
        return [kw.strip() for kw in v]
    
    @field_validator('confidence')
    @classmethod
    def round_confidence(cls, v: float) -> float:
        """置信度保留两位小数"""
        return round(v, 2)

# 展示 JSON Schema（Pydantic v2 自动生成）
print("=== CustomerFeedback JSON Schema ===")
print(json.dumps(CustomerFeedback.model_json_schema(), indent=2, ensure_ascii=False))

# 测试正常数据和错误数据
print("\n=== 验证测试 ===")

# 有效数据
valid_data = {
    "sentiment": "negative",
    "confidence": 0.95,
    "summary": "用户对响应速度非常不满，认为等待时间过长",
    "keywords": ["响应速度", "等待时间", "不满"],
    "urgency": "high",
    "rating": 1,
}
try:
    fb = CustomerFeedback(**valid_data)
    print(f"✓ 有效数据解析成功: {fb.model_dump()}")
except ValidationError as e:
    print(f"✗ 验证失败: {e}")

# 无效数据（confidence 超出范围）
invalid_data = {
    "sentiment": "positive",
    "confidence": 1.5,  # 超出 0-1 范围
    "summary": "好",  # 太短
    "keywords": [],  # 空列表
    "urgency": "medium",
}
try:
    fb = CustomerFeedback(**invalid_data)
    print(f"✓ 数据解析成功: {fb.model_dump()}")
except ValidationError as e:
    print(f"✗ 验证失败（预期）:")
    for err in e.errors():
        print(f"  - {err['loc']}: {err['msg']}")

---
## 2. 多级回退解析器 (Multi-Level Fallback Parser)

### 核心设计理念
LLM 的输出不可靠。需要"信任但要验证"的策略 — 优先使用最严格的解析器，逐级回退到更宽松的策略。

```
Level 1: json.loads()         ← 最严格，期望纯 JSON
  ↓ 失败
Level 2: 正则提取            ← 从 markdown 代码块提取 JSON
  ↓ 失败
Level 3: JSON 修复           ← 修复常见错误（尾逗号、单引号）
  ↓ 失败
Level 4: 重试 + 修正提示     ← 让 LLM 重新生成
  ↓ 失败
Level 5: 返回默认值/抛出异常  ← 最终兜底
```

In [ ]:
# ============================================================
# 多级回退解析器实现
# ============================================================

class RobustJSONParser:
    """
    多级回退 JSON 解析器。
    
    What: 对 LLM 输出执行多层解析尝试，最大化 JSON 提取成功率。
    Why: LLM 输出经常包含 markdown 包装、格式错误、多余文本等，需要鲁棒解析。
    When: 任何需要从 LLM 输出中提取结构化 JSON 的场景。
    """
    
    def __init__(self, max_retries: int = 2):
        self.max_retries = max_retries
        self._parse_log: list[dict] = []  # 解析日志
    
    def parse(self, raw_output: str, schema: type[BaseModel] | None = None) -> dict | BaseModel:
        """
        多级回退解析原始 LLM 输出。
        
        Args:
            raw_output: LLM 原始文本输出
            schema: 可选的 Pydantic 模型，用于最终验证
        Returns:
            解析后的 dict 或 Pydantic 模型实例
        Raises:
            ValueError: 所有回退层均失败
        """
        self._parse_log = []
        
        # Level 1: 直接 json.loads
        result = self._try_json_loads(raw_output)
        if result is not None:
            return self._validate_and_return(result, schema)
        
        # Level 2: 正则提取 JSON 块
        result = self._try_regex_extract(raw_output)
        if result is not None:
            return self._validate_and_return(result, schema)
        
        # Level 3: JSON 修复
        result = self._try_json_repair(raw_output)
        if result is not None:
            return self._validate_and_return(result, schema)
        
        # Level 4: 更激进的正则提取（键值对提取）
        result = self._try_kv_extraction(raw_output)
        if result is not None:
            return self._validate_and_return(result, schema)
        
        raise ValueError(f"所有 {len(self._parse_log)} 层回退解析均失败")
    
    def _try_json_loads(self, text: str) -> dict | None:
        """Level 1: 直接 JSON 解析"""
        try:
            result = json.loads(text.strip())
            self._parse_log.append({"level": 1, "method": "json.loads", "success": True})
            return result
        except json.JSONDecodeError as e:
            self._parse_log.append({"level": 1, "method": "json.loads", "success": False, "error": str(e)})
            return None
    
    def _try_regex_extract(self, text: str) -> dict | None:
        """
        Level 2: 正则提取 markdown 代码块中的 JSON。
        
        支持格式:
        - ```json ... ```
        - ```JSON ... ```
        - {...}（直接匹配 JSON 对象）
        """
        # 模式1: markdown 代码块
        patterns = [
            r'```(?:json|JSON)?\s*\n?(.*?)\n?```',  # 代码块
            r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',     # 简单 JSON 对象
        ]
        
        for i, pattern in enumerate(patterns):
            matches = re.findall(pattern, text, re.DOTALL)
            for match in matches:
                try:
                    result = json.loads(match.strip())
                    self._parse_log.append({
                        "level": 2, "method": f"regex_pattern_{i}", "success": True
                    })
                    return result
                except json.JSONDecodeError:
                    continue
        
        self._parse_log.append({"level": 2, "method": "regex", "success": False})
        return None
    
    def _try_json_repair(self, text: str) -> dict | None:
        """
        Level 3: 修复常见 JSON 错误。
        
        What: 自动修复尾逗号、单引号、无引号键等常见问题。
        Why: LLM 经常产生'差不多'的 JSON，微小的语法错误导致解析失败。
        When: 正则提取失败后。
        """
        # 先提取看起来像 JSON 的部分
        json_like = self._extract_json_like(text)
        if not json_like:
            self._parse_log.append({"level": 3, "method": "repair", "success": False})
            return None
        
        # 修复步骤
        fixed = json_like
        
        # 1. 替换单引号为双引号（但保留字符串内的转义单引号）
        fixed = re.sub(r"(?<!\\)'(?=.*?)", '"', fixed)
        
        # 2. 移除尾随逗号（对象和数组中的）
        fixed = re.sub(r',(\s*[}\]])', r'\1', fixed)
        
        # 3. 为无引号键添加引号
        fixed = re.sub(r'([{,]\s*)([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', fixed)
        
        # 4. 处理 None/True/False 的 Python 写法 -> JSON 写法
        fixed = fixed.replace("None", "null").replace("True", "true").replace("False", "false")
        
        try:
            result = json.loads(fixed)
            self._parse_log.append({"level": 3, "method": "repair", "success": True})
            return result
        except json.JSONDecodeError:
            self._parse_log.append({"level": 3, "method": "repair", "success": False})
            return None
    
    def _try_kv_extraction(self, text: str) -> dict | None:
        """
        Level 4: 最后兜底 — 键值对正则提取。
        
        What: 从任意文本中提取 "key": value 模式。
        Why: 当 LLM 完全未按 JSON 格式输出时，仍能回收部分结构化数据。
        When: 所有 JSON 解析方法均失败后。
        """
        result = {}
        # 匹配 "key": value, "key": "value", "key": [ ... ]
        kv_pattern = r'"([^"]+)"\s*:\s*("[^"]*"|[^,}\]]+)'
        matches = re.findall(kv_pattern, text)
        
        if not matches:
            self._parse_log.append({"level": 4, "method": "kv_extraction", "success": False})
            return None
        
        for key, value in matches:
            # 清理 value
            value = value.strip()
            # 尝试解析为 Python 原生类型
            try:
                result[key] = json.loads(value)
            except (json.JSONDecodeError, ValueError):
                # 去除可能的引号包装
                if value.startswith('"') and value.endswith('"'):
                    result[key] = value[1:-1]
                else:
                    result[key] = value
        
        self._parse_log.append({"level": 4, "method": "kv_extraction", "success": True})
        return result
    
    def _extract_json_like(self, text: str) -> str | None:
        """从文本中提取看起来像 JSON 的部分"""
        # 查找第一个 { 和最后一个 }
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and start < end:
            return text[start:end + 1]
        return None
    
    def _validate_and_return(
        self, data: dict, schema: type[BaseModel] | None = None
    ) -> dict | BaseModel:
        """
        使用 Pydantic Schema 验证最终解析结果。
        
        What: 将解析出的 dict 通过 Pydantic 模型验证，确保字段类型和约束。
        Why: 即使 JSON 解析成功，字段值可能不满足业务约束。
        When: schema 参数不为 None 时。
        """
        if schema is None:
            return data
        
        try:
            validated = schema(**data)
            return validated
        except ValidationError as e:
            # Pydantic 验证失败：返回未验证的 dict 但附加错误信息
            self._parse_log.append({
                "level": "validation", "method": "pydantic",
                "success": False, "error": str(e)
            })
            return data  # 返回原始 dict 而非抛出异常

# 创建全局解析器实例
robust_parser = RobustJSONParser()

print("RobustJSONParser 已实例化")

In [ ]:
# ============================================================
# 测试多级回退解析器
# ============================================================

test_outputs = [
    # 正常 JSON
    ('{"sentiment": "positive", "confidence": 0.95, "summary": "产品非常好用", "keywords": ["好用", "推荐"]}',
     "纯净 JSON"),
    # Markdown 代码块
    ('```json\n{"sentiment": "negative", "confidence": 0.88, "summary": "太慢", "keywords": ["速度"]}\n```',
     "Markdown 代码块"),
    # 有尾随逗号
    ('{"sentiment": "positive", "confidence": 0.8, "summary": "不错", "keywords": ["ok"],}',
     "尾随逗号"),
    # 单引号
    ('{\'sentiment\': \'negative\', \'confidence\': 0.5, \'summary\': \'一般\', \'keywords\': []}',
     "单引号"),
    # 有额外文本
    ('分析结果如下：该用户整体满意。{"sentiment": "positive", "confidence": 0.9, "summary": "满意", "keywords": ["满意"]}以上为分析结果。',
     "JSON 嵌入文本"),
]

print("=== 解析器回退测试 ===\n")
for raw_output, description in test_outputs:
    parser = RobustJSONParser()
    try:
        # 使用简化 schema 测试
        class SimpleFeedback(BaseModel):
            sentiment: str
            confidence: float = Field(ge=0, le=1)
            summary: str
            keywords: list[str]
        
        result = parser.parse(raw_output, SimpleFeedback)
        
        # 确定使用了哪一层
        levels_used = [log['level'] for log in parser._parse_log if log['success']]
        print(f"✓ {description}")
        print(f"  解析层级: {levels_used}")
        if isinstance(result, BaseModel):
            print(f"  结果: sentiment={result.sentiment}, confidence={result.confidence}")
        else:
            print(f"  结果: {result}")
        print()
    except ValueError as e:
        print(f"✗ {description}: {e}\n")

---
## 3. 专用解析器集合

不同的输出格式需要不同的解析策略。以下是常用的解析器类型。

In [ ]:
# ============================================================
# 专用解析器集合
# ============================================================

class OutputParser:
    """
    输出解析器基类。
    
    What: 为不同类型的输出格式定义统一的解析接口。
    Why: 策略模式 — 不同场景使用不同解析器，但对外接口一致。
    When: 构建支持多种输出格式的 LLM 应用时。
    """
    
    def parse(self, raw_output: str) -> Any:
        raise NotImplementedError("子类必须实现 parse 方法")


class CommaSeparatedListParser(OutputParser):
    """
    逗号分隔列表解析器。
    
    What: 将 "A, B, C" 格式的字符串解析为列表。
    Why: 关键词、标签等常以逗号分隔形式出现。
    When: 输出是扁平列表时（分类标签、关键词、特性列表）。
    """
    
    def __init__(self, strip: bool = True, deduplicate: bool = True):
        self.strip = strip
        self.deduplicate = deduplicate
    
    def parse(self, raw_output: str) -> list[str]:
        """
        解析逗号分隔字符串为列表。
        
        支持多种分隔符：逗号(,)、中文逗号(，)、分号(;)、顿号(、)
        """
        # 统一分隔符
        text = raw_output.replace("，", ",").replace("；", ";").replace("、", ",")
        
        # 按常见分隔符拆分
        items = re.split(r'[,;|\n]+', text)
        
        # 清理
        result = []
        for item in items:
            item = item.strip()
            # 移除编号前缀（如 "1. ", "- ", "* "）
            item = re.sub(r'^[\d]+[\.\)、]\s*', '', item)
            item = re.sub(r'^[-*•]\s*', '', item)
            if item:  # 排除空字符串
                result.append(item)
        
        # 去重（保持顺序）
        if self.deduplicate:
            seen = set()
            result = [x for x in result if not (x.lower() in seen or seen.add(x.lower()))]
        
        return result


class EnumParser(OutputParser):
    """
    枚举解析器：将自由文本匹配到预定义的枚举值。
    
    What: 使用模糊匹配将 LLM 输出映射到枚举成员。
    Why: LLM 可能不精确地输出枚举值（英文、大小写、同义词）。
    When: 输出被约束为一组有限值（分类标签、意图、优先级）。
    """
    
    def __init__(self, enum_class: type[Enum], fuzzy: bool = True):
        self.enum_class = enum_class
        self.fuzzy = fuzzy
        # 构建别名映射
        self._aliases: dict[str, str] = self._build_aliases()
    
    def parse(self, raw_output: str) -> Enum:
        """将文本输出解析为枚举值"""
        text = raw_output.strip().lower()
        
        # 精确匹配
        for member in self.enum_class:
            if text == member.value.lower():
                return member
        
        # 模糊匹配（通过别名映射）
        if self.fuzzy and text in self._aliases:
            canon = self._aliases[text]
            for member in self.enum_class:
                if member.value.lower() == canon:
                    return member
        
        # 子串匹配
        for member in self.enum_class:
            if member.value.lower() in text:
                return member
        
        raise ValueError(f"无法将 '{raw_output}' 映射到 {self.enum_class.__name__}")
    
    def _build_aliases(self) -> dict[str, str]:
        """构建常见别名的映射表"""
        alias_map = {
            # 通用映射
            "good": "positive", "great": "positive", "excellent": "positive",
            "bad": "negative", "poor": "negative", "terrible": "negative",
            "ok": "neutral", "average": "neutral", "normal": "neutral",
            # 优先级映射
            "urgent": "critical", "emergency": "critical", "severe": "critical",
            "important": "high", "moderate": "medium", "minor": "low",
        }
        return alias_map


class PydanticParser(OutputParser):
    """
    Pydantic 解析器：使用 Pydantic 模型验证和强制类型转换。
    
    What: 将 LLM JSON 输出解析为类型安全的 Pydantic 模型实例。
    Why: 提供完整的类型验证、默认值和自定义验证器。
    When: 输出结构复杂，需要强类型保证的场景。
    """
    
    def __init__(self, schema: type[BaseModel], use_robust_parser: bool = True):
        self.schema = schema
        self.robust_parser = RobustJSONParser() if use_robust_parser else None
    
    def parse(self, raw_output: str) -> BaseModel:
        """解析并验证为 Pydantic 模型"""
        if self.robust_parser:
            result = self.robust_parser.parse(raw_output, self.schema)
            if isinstance(result, BaseModel):
                return result
            # 否则手动构造
            return self.schema(**result)
        else:
            data = json.loads(raw_output)
            return self.schema(**data)


# 测试各解析器
print("=== CommaSeparatedListParser ===")
csl = CommaSeparatedListParser()
result = csl.parse("Python, JavaScript, TypeScript, python,  Rust  ")
print(f"解析结果: {result}")

print("\n=== EnumParser ===")
ep = EnumParser(Priority)
print(f"'urgent'   → {ep.parse('urgent')}")
print(f"'moderate' → {ep.parse('moderate')}")
print(f"'high'     → {ep.parse('high')}")

print("\n=== PydanticParser ===")
pp = PydanticParser(CustomerFeedback)
raw = '{"sentiment": "positive", "confidence": 0.92, "summary": "产品很好", "keywords": ["好"], "rating": 5}'
feedback = pp.parse(raw)
print(f"解析结果: {feedback.model_dump()}")
print(f"类型: {type(feedback).__name__}")

---
## 4. OpenAI Structured Outputs

### 原理
OpenAI 的 `response_format` 参数允许在 API 调用时指定输出 JSON Schema，模型会保证输出符合 schema 约束（通过 `strict` 模式）。

### 优势
1. **100% 符合 Schema**：在 `strict: true` 模式下保证
2. **无需回退解析器**：输出永远是有效的 JSON
3. **减少延迟**：不需要重试循环

In [ ]:
# ============================================================
# OpenAI Structured Outputs 演示（兼容模式）
# ============================================================

from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    base_url=os.environ.get("OPENAI_BASE_URL", None),
)

class StructuredOutputDemo:
    """
    使用 OpenAI Structured Outputs 的结构化输出演示。
    
    What: 展示如何通过 response_format 参数直接获取 Pydantic 兼容的结构化输出。
    Why: 这是最可靠的结构化输出方式，无需解析器。
    When: 使用支持 Structured Outputs 的模型（gpt-4o 系列）时。
    """
    
    @staticmethod
    def call_with_pydantic(
        system_prompt: str,
        user_prompt: str,
        response_model: type[BaseModel],
        model: str = "gpt-4o-mini",
    ) -> BaseModel:
        """
        使用 response_format 强制结构化输出。
        
        注意：此方法适用于 OpenAI SDK v1.x 及以上版本。
        对于不支持原生 Structured Outputs 的 API 端点，
        可以通过 json_mode + Pydantic 解析作为降级方案。
        """
        import json as json_module
        
        # 方案 A: 使用原生 Structured Outputs (OpenAI SDK v1.40+)
        # 如果 API 支持，可以直接：
        # response = client.beta.chat.completions.parse(
        #     model=model,
        #     messages=[...],
        #     response_format=response_model,
        # )
        
        # 方案 B: 使用 JSON mode + Schema（兼容性更好的降级方案）
        schema = response_model.model_json_schema()
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt + f"\n\n你必须以 JSON 格式返回，符合以下 schema：\n{json_module.dumps(schema, ensure_ascii=False)}"},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        
        raw = response.choices[0].message.content or "{}"
        parser = RobustJSONParser()
        result = parser.parse(raw, response_model)
        
        if isinstance(result, BaseModel):
            return result
        return response_model(**result)

# 演示
print("=== OpenAI Structured Outputs 演示 ===")

from pydantic import BaseModel, Field

class ProductReview(BaseModel):
    """产品评价结构化模型"""
    product_name: str = Field(description="产品名称")
    rating: int = Field(ge=1, le=5, description="评分 1-5")
    pros: list[str] = Field(description="优点列表")
    cons: list[str] = Field(description="缺点列表")
    recommend: bool = Field(description="是否推荐")
    summary: str = Field(min_length=10, description="总体评价")

try:
    result = StructuredOutputDemo.call_with_pydantic(
        system_prompt="你是一位产品评价分析助手。分析用户评论并提取结构化信息。",
        user_prompt="评论：AirPods Pro 2代降噪效果真的很惊艳，地铁上几乎听不到噪音。音质也不错。不过价格确实有点贵，充电盒容易划伤。总体来说推荐购买。",
        response_model=ProductReview,
    )
    print(f"产品: {result.product_name}")
    print(f"评分: {result.rating}/5")
    print(f"优点: {result.pros}")
    print(f"缺点: {result.cons}")
    print(f"推荐: {'是' if result.recommend else '否'}")
    print(f"总结: {result.summary}")
except Exception as e:
    print(f"API 调用失败（检查 Key）: {e}")
    print("提示: 导出环境变量 OPENAI_API_KEY 后重试")

---
## 5. Instructor 库替代方案

Instructor (https://github.com/jxnl/instructor) 是一个流行的结构化输出库，提供了比原生 OpenAI 更简洁的 API。以下展示自制轻量级替代方案，避免引入额外依赖。

In [ ]:
# ============================================================
# 轻量级 Instructor 替代实现
# ============================================================

class LightweightInstructor:
    """
    轻量级结构化输出提取器（Instructor 风格 API）。
    
    What: 提供类似 Instructor 库的 from_openai -> Pydantic 工作流。
    Why: Instructor 库很优秀，但依赖重。本实现仅用 OpenAI SDK + Pydantic。
    When: 不想引入 Instructor 依赖，但需要类似的使用体验。
    """
    
    def __init__(self, client: OpenAI | None = None):
        self.client = client or OpenAI(
            api_key=os.environ.get("OPENAI_API_KEY", "your-api-key"),
            base_url=os.environ.get("OPENAI_BASE_URL", None),
        )
        self.max_retries = 3
        self.parser = RobustJSONParser()
    
    def chat_completion(
        self,
        response_model: type[BaseModel],
        messages: list[dict],
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        max_retries: int | None = None,
    ) -> BaseModel:
        """
        发送请求并自动解析为 Pydantic 模型。
        
        Args:
            response_model: 目标 Pydantic 模型类
            messages: 消息列表
            model: 模型名称
            temperature: 温度参数
            max_retries: 最大重试次数
        Returns:
            已验证的 Pydantic 模型实例
        """
        retries = max_retries if max_retries is not None else self.max_retries
        
        # 注入 Schema 到 System Message
        schema_str = json.dumps(response_model.model_json_schema(), ensure_ascii=False)
        schema_instruction = f"\n\n你必须以 JSON 格式返回，严格遵守以下 Schema：\n{schema_str}\n\n只返回 JSON，不要包含任何其他文本。"
        
        for msg in messages:
            if msg["role"] == "system":
                msg["content"] += schema_instruction
                break
        else:
            messages.insert(0, {"role": "system", "content": schema_instruction})
        
        last_error = None
        for attempt in range(retries):
            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature,
                    response_format={"type": "json_object"},
                )
                raw = response.choices[0].message.content or "{}"
                result = self.parser.parse(raw, response_model)
                
                if isinstance(result, BaseModel):
                    return result
                # 否则用 dict 构造
                return response_model(**result)
            except (json.JSONDecodeError, ValidationError, ValueError) as e:
                last_error = e
                # 添加错误反馈到消息中，帮助模型修正
                messages.append({"role": "assistant", "content": raw if 'raw' in dir() else ""})
                messages.append({
                    "role": "user",
                    "content": f"你的输出不符合要求的格式。错误：{e}。请严格按照 JSON Schema 重新输出。"
                })
                continue
        
        raise ValueError(f"尝试 {retries} 次后仍失败: {last_error}")

li = LightweightInstructor()
print("LightweightInstructor 已实例化")
print("使用方法: li.chat_completion(response_model=MyModel, messages=[...], model='gpt-4o')")

---
## 6. 优雅降级策略 (Graceful Degradation)

当结构化输出失败时，系统不应崩溃。以下是优雅降级的最佳实践。

In [ ]:
# ============================================================
# 优雅降级策略
# ============================================================

from dataclasses import dataclass, field as dc_field
from datetime import datetime, timezone

@dataclass
class DegradationEvent:
    """降级事件记录：每次结构化输出失败时记录详情"""
    timestamp: datetime
    level: str  # "primary", "fallback_1", "fallback_2", "fallback_3", "failed"
    error_message: str
    raw_output_preview: str

@dataclass
class ParsedWithMeta:
    """
    带元数据的结果封装。
    
    What: 不仅返回解析结果，还附带解析过程信息。
    Why: 监控和调试需要了解解析是否使用了回退策略。
    When: 生产环境中需要可观测性的场景。
    """
    data: Any  # 解析结果
    success: bool  # 是否成功
    level_used: str  # 使用的解析层
    confidence: float  # 置信度（1.0 = 主解析器成功, 0.5 = 回退, 0.0 = 失败）
    degradation_events: list[DegradationEvent] = dc_field(default_factory=list)
    raw_output: str = ""


class GracefulStructuredExtractor:
    """
    带优雅降级的结构化提取器。
    
    What: 包装 RobustJSONParser，添加降级监控和默认值填充。
    Why: 生产环境中不应因解析失败而中断流程。
    When: 任何不能因为解析失败而阻塞的系统。
    """
    
    def __init__(self):
        self.parser = RobustJSONParser()
        self.degradation_log: list[DegradationEvent] = []
    
    def extract(
        self,
        raw_output: str,
        schema: type[BaseModel],
        defaults: dict | None = None,
    ) -> ParsedWithMeta:
        """
        提取结构化数据，失败时使用默认值。
        
        Args:
            raw_output: LLM 原始输出
            schema: 期望的 Pydantic 模型
            defaults: 失败时使用的默认值（不提供则创建空实例）
        Returns:
            ParsedWithMeta 对象
        """
        events: list[DegradationEvent] = []
        confidence = 1.0
        level = "primary"
        
        try:
            result = self.parser.parse(raw_output, schema)
            if isinstance(result, BaseModel):
                return ParsedWithMeta(
                    data=result, success=True, level_used=level,
                    confidence=1.0, raw_output=raw_output
                )
            # dict -> Pydantic
            validated = schema(**result)
            return ParsedWithMeta(
                data=validated, success=True, level_used=level,
                confidence=0.9, raw_output=raw_output
            )
        except ValueError as e:
            events.append(DegradationEvent(
                timestamp=datetime.now(timezone.utc),
                level="primary", error_message=str(e),
                raw_output_preview=raw_output[:200]
            ))
        
        # 尝试仅提取部分字段
        try:
            partial = {}
            field_names = list(schema.model_fields.keys())
            for field_name in field_names:
                pattern = rf'"{field_name}"\s*:\s*([^,}}\]]+)'
                match = re.search(pattern, raw_output)
                if match:
                    partial[field_name] = match.group(1).strip().strip('"')
            
            if partial:
                # 使用 Pydantic 默认值填充缺失字段
                filled = self._fill_defaults(schema, partial, defaults)
                return ParsedWithMeta(
                    data=filled, success=True, level_used="partial",
                    confidence=0.5, degradation_events=events, raw_output=raw_output
                )
        except Exception as e:
            events.append(DegradationEvent(
                timestamp=datetime.now(timezone.utc),
                level="partial", error_message=str(e),
                raw_output_preview=raw_output[:200]
            ))
        
        # 完全回退：返回默认值
        fallback = self._fill_defaults(schema, {}, defaults)
        events.append(DegradationEvent(
            timestamp=datetime.now(timezone.utc),
            level="failed", error_message="所有解析层失败，使用默认值",
            raw_output_preview=raw_output[:200]
        ))
        
        self.degradation_log.extend(events)
        
        return ParsedWithMeta(
            data=fallback, success=False, level_used="failed",
            confidence=0.0, degradation_events=events, raw_output=raw_output
        )
    
    def _fill_defaults(
        self,
        schema: type[BaseModel],
        partial: dict,
        overrides: dict | None = None,
    ) -> BaseModel:
        """使用字段默认值或提供的默认值填充缺失字段"""
        filled = {}
        for field_name, field_info in schema.model_fields.items():
            if field_name in partial:
                filled[field_name] = partial[field_name]
            elif overrides and field_name in overrides:
                filled[field_name] = overrides[field_name]
            elif field_info.default is not None:
                filled[field_name] = field_info.default
            else:
                # 根据类型设置默认值
                ftype = field_info.annotation
                if ftype is str:
                    filled[field_name] = "未知"
                elif ftype is int:
                    filled[field_name] = 0
                elif ftype is float:
                    filled[field_name] = 0.0
                elif ftype is bool:
                    filled[field_name] = False
                elif get_origin(ftype) is list:
                    filled[field_name] = []
                else:
                    filled[field_name] = None
        
        try:
            return schema(**filled)
        except ValidationError:
            # 如果类型不匹配，直接返回 partial
            return schema.construct(**filled) if hasattr(schema, 'construct') else None


# 测试优雅降级
extractor = GracefulStructuredExtractor()

print("=== 测试1: 正常输出 ===")
normal_output = '{"sentiment": "positive", "confidence": 0.88, "summary": "很好", "keywords": ["好"]}'
result = extractor.extract(normal_output, CustomerFeedback)
print(f"成功: {result.success}, 层级: {result.level_used}, 置信度: {result.confidence}")
if isinstance(result.data, CustomerFeedback):
    print(f"情感: {result.data.sentiment}")

print("\n=== 测试2: 严重损坏的输出 ===")
broken_output = "这个的反馈就是还可以吧我觉得没有太大的感觉就是这样了"
result = extractor.extract(broken_output, CustomerFeedback)
print(f"成功: {result.success}, 层级: {result.level_used}, 置信度: {result.confidence}")
print(f"降级事件数: {len(result.degradation_events)}")
for event in result.degradation_events:
    print(f"  [{event.level}] {event.error_message}")

print("\n优雅降级提取器测试完成！")

## 本节小结

1. **Pydantic 模型** 是结构化的"契约"，定义期望的输出格式和约束
2. **多级回退解析器** 确保在 LLM 输出不完美时仍能提取有用数据
3. **专用解析器**（Enum、CSL、Pydantic）覆盖不同的输出格式需求
4. **OpenAI Structured Outputs** 是最可靠的方式，但需要模型和 API 支持
5. **优雅降级** 是生产系统的必备策略，失败不崩溃而是返回部分结果 + 置信度